In [1]:
%%capture
# %% [Cell 1] Imports

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TF C++ logs (INFO, WARNING, ERROR)
import warnings
warnings.filterwarnings('ignore')  # Suppress Python warnings

import tensorflow as tf
tf.get_logger().setLevel('ERROR')  # Suppress TF Python-level logs
from tensorflow.keras.layers import Conv2D, Conv3D, Flatten, Dense, Reshape, BatchNormalization
from tensorflow.keras.layers import Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score
from scipy.io import loadmat
from scipy.signal import savgol_filter
import numpy as np
import matplotlib.pyplot as plt
import os
import gc
import pandas as pd
from pathlib import Path



E0000 00:00:1775747097.556882      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775747097.607001      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775747098.004972      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775747098.005010      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775747098.005013      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775747098.005016      24 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# %% [Cell 1b] Restore Previous Results (Kaggle)
import shutil
from pathlib import Path

# Move previous results into the working directory so the script knows to skip them!
old_results = Path('/kaggle/input/your-previous-dataset-name/CNN_Results_IP')
working_results = Path('/kaggle/working/CNN_Results_IP')

if old_results.exists():
    shutil.copytree(old_results, working_results, dirs_exist_ok=True)
    print("Previous checkpoints restored!")



In [3]:
# %% [Cell 2] Load Data
DATASET_DIR = Path(r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_WST_Pipeline/dataset")

# Auto-detect Kaggle Environment
if Path('/kaggle/input').exists():
    print("Detected Kaggle Environment. Adjusting paths...")
    found_files = list(Path('/kaggle/input').rglob('Indian_pines_corrected.mat'))
    if found_files:
        DATASET_DIR = found_files[0].parent
        print(f"Found dataset at: {DATASET_DIR}")
    else:
        print("Warning: Could not find Indian_pines_corrected.mat in /kaggle/input")

print(f"Loading data from {DATASET_DIR}...")

data_path = DATASET_DIR / 'Indian_pines_corrected.mat'
gt_path = DATASET_DIR / 'Indian_pines_gt.mat'

if not data_path.exists():
    raise FileNotFoundError(f"File not found: {data_path}")

X_raw = loadmat(str(data_path))['indian_pines_corrected']
y_raw = loadmat(str(gt_path))['indian_pines_gt']

print(f"Data Loaded. Shape: {X_raw.shape}")



Detected Kaggle Environment. Adjusting paths...
Found dataset at: /kaggle/input/datasets/hosseinkaka1997/indianpines
Loading data from /kaggle/input/datasets/hosseinkaka1997/indianpines...
Data Loaded. Shape: (145, 145, 200)


In [4]:
# %% [Cell 3] Constants
test_ratio = 0.8
DEFAULT_PIXELSIZE = 25

In [5]:
# %% [Cell 4] Preprocessing Functions
def padWithZeros(X, margin=2):
    newX = np.zeros((X.shape[0] + 2 * margin, X.shape[1] + 2* margin, X.shape[2]))
    x_offset = margin
    y_offset = margin
    newX[x_offset:X.shape[0] + x_offset, y_offset:X.shape[1] + y_offset, :] = X
    return newX

def select_wavelengths(X, selected_indices):
    print(f"Selecting {len(selected_indices)} bands...")
    return X[:, :, selected_indices]

def apply_snv(X):
    print("Applying SNV...")
    mean_vec = np.mean(X, axis=2, keepdims=True)
    std_vec = np.std(X, axis=2, keepdims=True)
    return (X - mean_vec) / (std_vec + 1e-8)

def apply_msc(X, reference=None):
    print("Applying MSC...")
    original_shape = X.shape
    X_flat = X.reshape(-1, original_shape[2])
    if reference is None:
        reference = np.mean(X_flat, axis=0)
    n_samples = X_flat.shape[0]
    X_msc = np.zeros_like(X_flat)
    for i in range(n_samples):
        fit = np.polyfit(reference, X_flat[i, :], 1, full=True)
        slope = fit[0][0]
        intercept = fit[0][1]
        X_msc[i, :] = (X_flat[i, :] - intercept) / slope
    return X_msc.reshape(original_shape)

def apply_sg(X, deriv=0, window_length=30, polyorder=2):
    print(f"Applying SG (w={window_length}, p={polyorder}, d={deriv})...")
    return savgol_filter(X, window_length=window_length, polyorder=polyorder, deriv=deriv, axis=2)

def preprocess_pipeline(X, pipeline_name):
    print(f"Preprocessing Pipeline: {pipeline_name}")
    X_proc = X.copy()
    if 'SG' in pipeline_name:
        deriv = 1 if 'SG1' in pipeline_name else 0 
        X_proc = apply_sg(X_proc, deriv=deriv, window_length=30, polyorder=2)
    if 'MSC' in pipeline_name:
        X_proc = apply_msc(X_proc)
    elif 'SNV' in pipeline_name or 'SVN' in pipeline_name:
        X_proc = apply_snv(X_proc)
    return X_proc

def createImageCubes(X, y, windowSize=5, removeZeroLabels=True):
    margin = int((windowSize - 1) / 2)
    zeroPaddedX = padWithZeros(X, margin=margin)
    if removeZeroLabels:
        r_indices, c_indices = np.nonzero(y)
        num_samples = len(r_indices)
    else:
        r_indices, c_indices = np.where(y >= 0)
        num_samples = len(r_indices)
    print(f"Generating {num_samples} patches (Optimized)...")
    patchesData = np.zeros((num_samples, windowSize, windowSize, X.shape[2]), dtype=np.float32)
    patchesLabels = np.zeros((num_samples))
    for i in range(num_samples):
        r = r_indices[i]
        c = c_indices[i]
        r_pad = r + margin
        c_pad = c + margin
        patch = zeroPaddedX[r_pad - margin:r_pad + margin + 1, c_pad - margin:c_pad + margin + 1]
        patchesData[i, :, :, :] = patch
        patchesLabels[i] = y[r, c]
    if removeZeroLabels:
        patchesLabels -= 1
    return patchesData, patchesLabels



In [6]:
# %% [Cell 5] Print Info
print(f"Original X shape: {X_raw.shape}, y shape: {y_raw.shape}")



Original X shape: (145, 145, 200), y shape: (145, 145)


In [7]:
# %% [Cell 6] Helper Functions
def get_indices_from_values(selected_values, all_wavelengths):
    indices = []
    for val in selected_values:
        idx = (np.abs(all_wavelengths - val)).argmin()
        indices.append(idx)
    return sorted(list(set(indices)))

# Load Reference Wavelengths
WAVELENGTH_FILE = r"C:\Users\hosse\Desktop\Thesis_Phase1_completed\HSI_WST_Pipeline\dataset\indianpines_wavelengths_200.csv"

if Path('/kaggle/input').exists():
    found_wl = list(Path('/kaggle/input').rglob('indianpines_wavelengths_200.csv'))
    if found_wl:
        WAVELENGTH_FILE = str(found_wl[0])
        print(f"Found Reference Wavelengths at: {WAVELENGTH_FILE}")

if os.path.exists(WAVELENGTH_FILE):
    all_wl = pd.read_csv(WAVELENGTH_FILE).iloc[:, 0].values
else:
    all_wl = np.arange(200)



Found Reference Wavelengths at: /kaggle/input/datasets/hosseinkaka1997/indianpines/indianpines_wavelengths_200.csv


In [8]:
# %% [Cell 7] Train and Evaluate Function
def train_and_evaluate(method_name, pipeline_name, selected_indices, X_raw, y_raw):
    print(f"\n{'='*50}")
    print(f"Starting Experiment: {method_name} ({pipeline_name})")
    print(f"{'='*50}")

    # --- 1. SET UP KAGGLE-SAFE DIRECTORY & CHECKPOINTING ---
    if Path('/kaggle/working').exists():
        RESULTS_DIR = Path('/kaggle/working/CNN_Results_IP')
    else:
        RESULTS_DIR = Path(r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_Fresh_Adaptation/CNN_Results_IP")
    
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    safe_method_name = f"{method_name}_{pipeline_name}".replace(" ", "_").replace("/", "-")
    
    # Check if this experiment has already finished in a previous run
    report_path = RESULTS_DIR / f"classification_report_{safe_method_name}.txt"
    if report_path.exists():
        print(f"✅ Experiment {safe_method_name} is already complete! Found existing results. Skipping to save time...")
        return # Exit the function early and move to the next experiment
    # -------------------------------------------------------

    # Apply Preprocessing
    if pipeline_name != 'RAW':
        X_processed = preprocess_pipeline(X_raw, pipeline_name)
    else:
        X_processed = X_raw.copy()

    # Select Wavelengths
    X = select_wavelengths(X_processed, selected_indices)
    del X_processed
    y = y_raw.copy()

    # Adaptive pixelsize based on band count (to fit in Kaggle RAM)
    n_bands = len(selected_indices)
    if n_bands > 80:
        pixelsize = 13
    elif n_bands > 40:
        pixelsize = 19
    else:
        pixelsize = DEFAULT_PIXELSIZE
    print(f"Using pixelsize={pixelsize} for {n_bands} bands")

    # Determine num_classes
    unique_classes = np.unique(y_raw)
    unique_classes = unique_classes[unique_classes != 0]
    num_classes = len(unique_classes)
    print(f"Detected {num_classes} classes for spatial split.")

    # Block-Based Spatial Train/Test Split
    y_train_map = np.zeros_like(y_raw)
    y_test_map = np.zeros_like(y_raw)
    
    H_img, W_img = y_raw.shape
    block_size = 50  
    
    for c in range(1, num_classes + 1):
        class_indices = np.argwhere(y_raw == c)
        if len(class_indices) == 0: continue
        class_blocks_set = set()
        for r, col in class_indices:
            class_blocks_set.add((r // block_size, col // block_size))
        class_blocks = list(class_blocks_set)
        np.random.shuffle(class_blocks)
        n_test = int(len(class_blocks) * test_ratio)
        if n_test == len(class_blocks) and len(class_blocks) > 1:
            n_test -= 1
        test_blocks_set = set(class_blocks[:n_test])
        for r, col in class_indices:
            block_id = (r // block_size, col // block_size)
            if block_id in test_blocks_set:
                y_test_map[r, col] = c
            else:
                y_train_map[r, col] = c

    # Generate cubes
    Xtrain, ytrain = createImageCubes(X, y_train_map, windowSize=pixelsize, removeZeroLabels=True)
    Xtest, ytest = createImageCubes(X, y_test_map, windowSize=pixelsize, removeZeroLabels=True)
    print(f"Train shapes: {Xtrain.shape}, Test shapes: {Xtest.shape}")

    # Validation Split
    Xtrain, Xvalid, ytrain, yvalid = train_test_split(Xtrain, ytrain, test_size=0.3333, random_state=42, stratify=ytrain)

    # StandardScaler (Fit on Train)
    print("Normalizing data (StandardScaler) - Fit on Train...")
    N_tr, H, W, C = Xtrain.shape
    scaler = StandardScaler()
    Xtrain = scaler.fit_transform(Xtrain.reshape(-1, C)).reshape(N_tr, H, W, C)
    N_te = Xtest.shape[0]
    Xtest = scaler.transform(Xtest.reshape(-1, C)).reshape(N_te, H, W, C)
    N_val = Xvalid.shape[0]
    Xvalid = scaler.transform(Xvalid.reshape(-1, C)).reshape(N_val, H, W, C)
    print("Normalization Complete.")

    # Verification
    current_bands = Xtrain.shape[3]
    expected_bands = len(selected_indices)
    assert current_bands == expected_bands

    # Reshape for CNN
    input_channels = Xtrain.shape[3]
    Xtrain = Xtrain.reshape(-1, pixelsize, pixelsize, input_channels, 1)
    Xtest = Xtest.reshape(-1, pixelsize, pixelsize, input_channels, 1)
    Xvalid = Xvalid.reshape(-1, pixelsize, pixelsize, input_channels, 1)

    # One-Hot Encoding
    ytrain = to_categorical(ytrain)
    ytest = to_categorical(ytest)
    yvalid = to_categorical(yvalid)
    num_classes = ytrain.shape[1]

    # Model Definition
    input_shape = (pixelsize, pixelsize, input_channels, 1)

    def build_adaptive_cnn(input_shape, num_classes):
        input_layer = Input(input_shape)
        depth = input_shape[2]
        k1 = 7 if depth >= 7 else (5 if depth >= 5 else 3)
        d1 = depth - (k1 - 1)
        k2 = 5 if d1 >= 5 else (3 if d1 >= 3 else 1)
        d2 = d1 - (k2 - 1)
        k3 = 3 if d2 >= 3 else 1
        x = Conv3D(filters=8, kernel_size=(3, 3, k1), activation='relu')(input_layer)
        x = Conv3D(filters=16, kernel_size=(3, 3, k2), activation='relu')(x)
        x = Conv3D(filters=32, kernel_size=(3, 3, k3), activation='relu')(x)
        s = x.shape
        x = Reshape((s[1], s[2], s[3] * s[4]))(x)
        x = Conv2D(filters=64, kernel_size=(3,3), activation='relu')(x)
        x = Flatten()(x)
        x = Dense(units=256, activation='relu')(x)
        x = Dropout(0.4)(x)
        x = Dense(units=128, activation='relu')(x)
        x = Dropout(0.4)(x)
        outputs = Dense(units=num_classes, activation='softmax')(x)
        model = Model(inputs=input_layer, outputs=outputs)
        return model

    model = build_adaptive_cnn(input_shape, num_classes)
    adam = Adam(learning_rate=0.001, decay=1e-06)
    model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])

    # --- 2. RESUME WEIGHTS IF KERNEL DIED MID-TRAINING ---
    checkpoint_path = RESULTS_DIR / f"best_model_{safe_method_name}.keras"
    if checkpoint_path.exists():
        print("Found partial training weights! Loading them before continuing...")
        model.load_weights(str(checkpoint_path))
    
    checkpoint = ModelCheckpoint(str(checkpoint_path), monitor='val_accuracy', verbose=0, save_best_only=True, mode='max')
    callbacks_list = [checkpoint]

    # Train with Data Augmentation
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    EPOCHS = 100
    n_bands = len(selected_indices)
    if n_bands > 80:
        BATCH_SIZE = 64
    elif n_bands > 40:
        BATCH_SIZE = 128
    else:
        BATCH_SIZE = 256
    
    Xtrain_aug = Xtrain.squeeze(-1)
    datagen = ImageDataGenerator(
        rotation_range=90,
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode='reflect'
    )
    train_gen = datagen.flow(Xtrain_aug, ytrain, batch_size=BATCH_SIZE)
    
    def expand_dims_generator(generator):
        for x_batch, y_batch in generator:
            yield np.expand_dims(x_batch, axis=-1), y_batch

    print(f"Training {safe_method_name} with Data Augmentation for {EPOCHS} epochs...")
    history = model.fit(
        expand_dims_generator(train_gen),
        steps_per_epoch=len(Xtrain) // BATCH_SIZE,
        validation_data=(Xvalid, yvalid),
        epochs=EPOCHS, 
        callbacks=callbacks_list, 
        verbose=1
    )

    # Evaluate
    model.load_weights(str(checkpoint_path))
    model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
    EVAL_BATCH = min(BATCH_SIZE, 64)
    loss, accuracy = model.evaluate(Xtest, ytest, batch_size=EVAL_BATCH, verbose=0)
    print(f"Test Accuracy ({safe_method_name}): {accuracy*100:.2f}%")

    # Reports
    Y_pred_test = model.predict(Xtest, batch_size=EVAL_BATCH, verbose=0)
    y_pred_test = np.argmax(Y_pred_test, axis=1)
    y_test_labels = np.argmax(ytest, axis=1)
    classification = classification_report(y_test_labels, y_pred_test)
    confusion = confusion_matrix(y_test_labels, y_pred_test)
    
    with open(report_path, 'w') as f:
        f.write(classification)
        f.write("\n\nConfusion Matrix:\n")
        f.write(str(confusion))
        f.write(f"\n\nTest Accuracy: {accuracy*100:.2f}%")

    # Plotting
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'Loss: {safe_method_name}')
    plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'Accuracy: {safe_method_name}')
    plt.legend(); plt.grid(True)
    plot_path = RESULTS_DIR / f"training_plot_{safe_method_name}.png"
    plt.savefig(plot_path)
    plt.close()
    print(f"Finished {safe_method_name}. Results saved.")

    # Memory cleanup
    del model, Xtrain, Xtest, Xvalid, ytrain, ytest, yvalid, Y_pred_test, Xtrain_aug
    tf.keras.backend.clear_session()
    gc.collect()
    print("Memory cleared.")



In [9]:
# %% [Cell 8] Load WST CSV Files
import pandas as pd
from pathlib import Path
local_wst_dir = Path(r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_Fresh_Adaptation/WST_Results/Indian_pines")

pipeline_dirs = ['10_SG_MSC', '10_SG_SVN', '10_SG1_MSC', '10_SG1_SVN']
all_experiments = {}

if Path('/kaggle/input').exists():
    print("Running in Kaggle environment...")
    kaggle_base_dir = Path('/kaggle/input')
    
    for pdir in pipeline_dirs:
        search_pattern = f"*{pdir}*_selected_wavelengths.csv"
        found_csvs = list(kaggle_base_dir.rglob(search_pattern))
        
        if found_csvs:
            df = pd.read_csv(found_csvs[0])
            all_experiments[pdir] = df
            print(f"\n{pdir}: {len(df)} methods loaded")
            if 'Method' in df.columns and 'N_Selected' in df.columns:
                print(df[['Method', 'N_Selected']].to_string())
            else:
                print("Columns 'Method' and 'N_Selected' are not in this CSV.")
        else:
            print(f"Warning: No CSV found for {pdir} in Kaggle!")

else:
    print("Running in Local environment...")
    for pdir in pipeline_dirs:
        csv_file = list((local_wst_dir / pdir).glob('*_selected_wavelengths.csv'))
        if csv_file:
            df = pd.read_csv(csv_file[0])
            all_experiments[pdir] = df
            print(f"\n{pdir}: {len(df)} methods loaded")
            if 'Method' in df.columns and 'N_Selected' in df.columns:
                print(df[['Method', 'N_Selected']].to_string())
        else:
            print(f"Warning: No CSV found in {local_wst_dir / pdir}")



Running in Kaggle environment...

10_SG_MSC: 4 methods loaded
         Method  N_Selected
0          BOSS         119
1          CARS          40
2       GA-iPLS          90
3  GA-iPLS_BOSS          16

10_SG_SVN: 4 methods loaded
         Method  N_Selected
0          BOSS          32
1          CARS          26
2       GA-iPLS          80
3  GA-iPLS_BOSS          26

10_SG1_MSC: 4 methods loaded
         Method  N_Selected
0          BOSS          15
1          CARS          25
2       GA-iPLS          90
3  GA-iPLS_BOSS          56

10_SG1_SVN: 4 methods loaded
         Method  N_Selected
0          BOSS           9
1          CARS           4
2       GA-iPLS          96
3  GA-iPLS_BOSS           7


In [10]:
# %% [Cell 8b] Load FCovSel CSV
local_fcovsel_csv = Path(r"c:/Users/hosse/Desktop/Thesis_Phase1_completed/HSI_Fresh_Adaptation/Indian_pines_all_FCovSel/Indian_pines_FCovSel_selected_wavelengths.csv")

if Path('/kaggle/input').exists():
    found_fcovsel = list(Path('/kaggle/input').rglob('Indian_pines_FCovSel_selected_wavelengths.csv'))
    if found_fcovsel:
        fcovsel_csv = found_fcovsel[0]
    else:
        fcovsel_csv = local_fcovsel_csv
else:
    fcovsel_csv = local_fcovsel_csv

if fcovsel_csv.exists():
    fcovsel_df = pd.read_csv(fcovsel_csv)
    print(f"\nFCovSel: {len(fcovsel_df)} pipelines loaded")
    print(fcovsel_df[['Pipeline', 'N_Selected']].to_string())
else:
    print(f"Warning: FCovSel CSV not found at {fcovsel_csv}")
    fcovsel_df = None




FCovSel: 5 pipelines loaded
  Pipeline  N_Selected
0      RAW          19
1   SG MSC          19
2   SG SNV          20
3  SG1 MSC          19
4  SG1 SNV          20


In [11]:
# %% [Cell 9] Helper: Parse WST Row
def parse_wst_row(row):
    method_name = row['Method']
    pipeline_name = row['Pipeline']
    bands_str = row['Selected_Wavelengths']
    bands_str = bands_str.replace('[', '').replace(']', '').replace('\n', '')
    selected_wavelengths = [float(x) for x in bands_str.split(',') if x.strip()]
    return selected_wavelengths, method_name, pipeline_name

# ============================================================
# Pipeline 1: 10_SG_MSC (4 methods)
# ============================================================



In [12]:
# %% [Cell 10] Experiment 1/16: BOSS (SG_MSC)
row = all_experiments['10_SG_MSC'].iloc[0]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: BOSS (10_SG_MSC)
Preprocessing Pipeline: 10_SG_MSC
Applying SG (w=30, p=2, d=0)...
Applying MSC...
Selecting 119 bands...
Using pixelsize=13 for 119 bands
Detected 16 classes for spatial split.
Generating 2824 patches (Optimized)...
Generating 7425 patches (Optimized)...
Train shapes: (2824, 13, 13, 119), Test shapes: (7425, 13, 13, 119)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.


I0000 00:00:1775747122.253674      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Training BOSS_10_SG_MSC with Data Augmentation for 100 epochs...
Epoch 1/100


I0000 00:00:1775747126.005824      68 service.cc:152] XLA service 0x7feb6800f730 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775747126.005882      68 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1775747126.464566      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


 1/29 ━━━━━━━━━━━━━━━━━━━━ 3:34 8s/step - accuracy: 0.0625 - loss: 2.7597

I0000 00:00:1775747131.708760      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


29/29 ━━━━━━━━━━━━━━━━━━━━ 18s 359ms/step - accuracy: 0.2465 - loss: 2.2933 - val_accuracy: 0.6444 - val_loss: 1.2162
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 11s 272ms/step - accuracy: 0.5706 - loss: 1.2788 - val_accuracy: 0.8185 - val_loss: 0.5660
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 8s 278ms/step - accuracy: 0.7331 - loss: 0.8047 - val_accuracy: 0.8450 - val_loss: 0.4266
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 8s 283ms/step - accuracy: 0.8194 - loss: 0.5205 - val_accuracy: 0.8843 - val_loss: 0.3208
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 8s 278ms/step - accuracy: 0.8657 - loss: 0.3922 - val_accuracy: 0.9299 - val_loss: 0.1939
Epoch 6/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 8s 283ms/step - accuracy: 0.9040 - loss: 0.2864 - val_accuracy: 0.9352 - val_loss: 0.1547
Epoch 7/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 8s 282ms/step - accuracy: 0.8987 - loss: 0.2578 - val_accuracy: 0.9851 - val_loss: 0.0640
Epoch 8/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 8s 278ms/step - accuracy: 0.9273 - loss: 0.2123 - val_accuracy: 0.96

In [13]:
# %% [Cell 11] Experiment 2/16: CARS (SG_MSC)
row = all_experiments['10_SG_MSC'].iloc[1]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: CARS (10_SG_MSC)
Preprocessing Pipeline: 10_SG_MSC
Applying SG (w=30, p=2, d=0)...
Applying MSC...
Selecting 40 bands...
Using pixelsize=25 for 40 bands
Detected 16 classes for spatial split.
Generating 3692 patches (Optimized)...
Generating 6557 patches (Optimized)...
Train shapes: (3692, 25, 25, 40), Test shapes: (6557, 25, 25, 40)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training CARS_10_SG_MSC with Data Augmentation for 100 epochs...
Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 24s 979ms/step - accuracy: 0.1585 - loss: 2.5507 - val_accuracy: 0.2299 - val_loss: 1.9877
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 15s 660ms/step - accuracy: 0.2704 - loss: 2.0267 - val_accuracy: 0.4533 - val_loss: 1.6317
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 770ms/step - accuracy: 0.3503 - loss: 1.8148 - val_accuracy: 0.6588 - val_loss: 1.2089
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 7s 785ms/step - accuracy: 0.5445 - loss: 1.3471 - val_accuracy: 0.7994 - val_loss

In [14]:
# %% [Cell 12] Experiment 3/16: GA-iPLS (SG_MSC)
row = all_experiments['10_SG_MSC'].iloc[2]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: GA-iPLS (10_SG_MSC)
Preprocessing Pipeline: 10_SG_MSC
Applying SG (w=30, p=2, d=0)...
Applying MSC...
Selecting 90 bands...
Using pixelsize=13 for 90 bands
Detected 16 classes for spatial split.
Generating 2693 patches (Optimized)...
Generating 7556 patches (Optimized)...
Train shapes: (2693, 13, 13, 90), Test shapes: (7556, 13, 13, 90)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training GA-iPLS_10_SG_MSC with Data Augmentation for 100 epochs...
Epoch 1/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 281ms/step - accuracy: 0.2365 - loss: 2.3926 - val_accuracy: 0.4555 - val_loss: 1.5564
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 9s 212ms/step - accuracy: 0.4723 - loss: 1.6655 - val_accuracy: 0.7038 - val_loss: 0.9843
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 215ms/step - accuracy: 0.6217 - loss: 1.0566 - val_accuracy: 0.8697 - val_loss: 0.5598
Epoch 4/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 216ms/step - accuracy: 0.7659 - loss: 0.7074 - val_accuracy: 0.88

In [15]:
# %% [Cell 13] Experiment 4/16: GA-iPLS_BOSS (SG_MSC)
row = all_experiments['10_SG_MSC'].iloc[3]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)

# ============================================================
# Pipeline 2: 10_SG_SVN (4 methods)
# ============================================================




Starting Experiment: GA-iPLS_BOSS (10_SG_MSC)
Preprocessing Pipeline: 10_SG_MSC
Applying SG (w=30, p=2, d=0)...
Applying MSC...
Selecting 16 bands...
Using pixelsize=25 for 16 bands
Detected 16 classes for spatial split.
Generating 4086 patches (Optimized)...
Generating 6163 patches (Optimized)...
Train shapes: (4086, 25, 25, 16), Test shapes: (6163, 25, 25, 16)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training GA-iPLS_BOSS_10_SG_MSC with Data Augmentation for 100 epochs...
Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 480ms/step - accuracy: 0.2004 - loss: 2.3682 - val_accuracy: 0.4824 - val_loss: 1.5740
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 293ms/step - accuracy: 0.4407 - loss: 1.6020 - val_accuracy: 0.6028 - val_loss: 1.1940
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 337ms/step - accuracy: 0.5919 - loss: 1.2531 - val_accuracy: 0.7988 - val_loss: 0.7393
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 339ms/step - accuracy: 0.6904 - loss: 0.9377 - val_accu

In [16]:
# %% [Cell 14] Experiment 5/16: BOSS (SG_SVN)
row = all_experiments['10_SG_SVN'].iloc[0]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: BOSS (10_SG_SVN)
Preprocessing Pipeline: 10_SG_SVN
Applying SG (w=30, p=2, d=0)...
Applying SNV...
Selecting 31 bands...
Using pixelsize=25 for 31 bands
Detected 16 classes for spatial split.
Generating 3704 patches (Optimized)...
Generating 6545 patches (Optimized)...
Train shapes: (3704, 25, 25, 31), Test shapes: (6545, 25, 25, 31)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training BOSS_10_SG_SVN with Data Augmentation for 100 epochs...
Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 20s 776ms/step - accuracy: 0.2169 - loss: 2.4976 - val_accuracy: 0.4721 - val_loss: 1.7997
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 13s 523ms/step - accuracy: 0.3935 - loss: 1.7398 - val_accuracy: 0.5579 - val_loss: 1.3343
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 5s 613ms/step - accuracy: 0.5850 - loss: 1.3170 - val_accuracy: 0.7781 - val_loss: 0.8804
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 5s 609ms/step - accuracy: 0.7258 - loss: 0.9083 - val_accuracy: 0.8057 - val_loss

In [17]:
# %% [Cell 15] Experiment 6/16: CARS (SG_SVN)
row = all_experiments['10_SG_SVN'].iloc[1]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: CARS (10_SG_SVN)
Preprocessing Pipeline: 10_SG_SVN
Applying SG (w=30, p=2, d=0)...
Applying SNV...
Selecting 26 bands...
Using pixelsize=25 for 26 bands
Detected 16 classes for spatial split.
Generating 3300 patches (Optimized)...
Generating 6949 patches (Optimized)...
Train shapes: (3300, 25, 25, 26), Test shapes: (6949, 25, 25, 26)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training CARS_10_SG_SVN with Data Augmentation for 100 epochs...
Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 16s 718ms/step - accuracy: 0.2115 - loss: 2.4344 - val_accuracy: 0.5364 - val_loss: 1.4186
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 9s 433ms/step - accuracy: 0.5703 - loss: 1.3366 - val_accuracy: 0.7773 - val_loss: 0.8327
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 537ms/step - accuracy: 0.7106 - loss: 0.9553 - val_accuracy: 0.8355 - val_loss: 0.5126
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 534ms/step - accuracy: 0.8059 - loss: 0.6358 - val_accuracy: 0.8700 - val_loss:

In [18]:
# %% [Cell 16] Experiment 7/16: GA-iPLS (SG_SVN)
row = all_experiments['10_SG_SVN'].iloc[2]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: GA-iPLS (10_SG_SVN)
Preprocessing Pipeline: 10_SG_SVN
Applying SG (w=30, p=2, d=0)...
Applying SNV...
Selecting 79 bands...
Using pixelsize=19 for 79 bands
Detected 16 classes for spatial split.
Generating 2987 patches (Optimized)...
Generating 7262 patches (Optimized)...
Train shapes: (2987, 19, 19, 79), Test shapes: (7262, 19, 19, 79)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training GA-iPLS_10_SG_SVN with Data Augmentation for 100 epochs...
Epoch 1/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 21s 650ms/step - accuracy: 0.2254 - loss: 2.4325 - val_accuracy: 0.4518 - val_loss: 1.6016
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 471ms/step - accuracy: 0.4376 - loss: 1.6225 - val_accuracy: 0.6817 - val_loss: 1.1211
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 517ms/step - accuracy: 0.6070 - loss: 1.1988 - val_accuracy: 0.8193 - val_loss: 0.6027
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 525ms/step - accuracy: 0.7561 - loss: 0.7364 - val_accuracy: 0.8

In [19]:
# %% [Cell 17] Experiment 8/16: GA-iPLS_BOSS (SG_SVN)
row = all_experiments['10_SG_SVN'].iloc[3]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)

# ============================================================
# Pipeline 3: 10_SG1_MSC (4 methods)
# ============================================================




Starting Experiment: GA-iPLS_BOSS (10_SG_SVN)
Preprocessing Pipeline: 10_SG_SVN
Applying SG (w=30, p=2, d=0)...
Applying SNV...
Selecting 26 bands...
Using pixelsize=25 for 26 bands
Detected 16 classes for spatial split.
Generating 2584 patches (Optimized)...
Generating 7665 patches (Optimized)...
Train shapes: (2584, 25, 25, 26), Test shapes: (7665, 25, 25, 26)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training GA-iPLS_BOSS_10_SG_SVN with Data Augmentation for 100 epochs...
Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 793ms/step - accuracy: 0.1698 - loss: 2.7257 - val_accuracy: 0.2169 - val_loss: 2.2923
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 420ms/step - accuracy: 0.3078 - loss: 2.2177 - val_accuracy: 0.5255 - val_loss: 1.5453
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 537ms/step - accuracy: 0.4758 - loss: 1.6801 - val_accuracy: 0.7413 - val_loss: 0.9605
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 547ms/step - accuracy: 0.6123 - loss: 1.2682 - val_accuracy: 0.7

In [20]:
# %% [Cell 18] Experiment 9/16: BOSS (SG1_MSC)
row = all_experiments['10_SG1_MSC'].iloc[0]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: BOSS (10_SG1_MSC)
Preprocessing Pipeline: 10_SG1_MSC
Applying SG (w=30, p=2, d=1)...
Applying MSC...
Selecting 15 bands...
Using pixelsize=25 for 15 bands
Detected 16 classes for spatial split.
Generating 3507 patches (Optimized)...
Generating 6742 patches (Optimized)...
Train shapes: (3507, 25, 25, 15), Test shapes: (6742, 25, 25, 15)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training BOSS_10_SG1_MSC with Data Augmentation for 100 epochs...
Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 11s 509ms/step - accuracy: 0.2345 - loss: 2.3707 - val_accuracy: 0.6424 - val_loss: 1.1809
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 336ms/step - accuracy: 0.6333 - loss: 1.1832 - val_accuracy: 0.7896 - val_loss: 0.6686
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 323ms/step - accuracy: 0.7620 - loss: 0.7598 - val_accuracy: 0.8725 - val_loss: 0.4377
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 324ms/step - accuracy: 0.8464 - loss: 0.5157 - val_accuracy: 0.9145 - val_lo

In [21]:
# %% [Cell 19] Experiment 10/16: CARS (SG1_MSC)
row = all_experiments['10_SG1_MSC'].iloc[1]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: CARS (10_SG1_MSC)
Preprocessing Pipeline: 10_SG1_MSC
Applying SG (w=30, p=2, d=1)...
Applying MSC...
Selecting 25 bands...
Using pixelsize=25 for 25 bands
Detected 16 classes for spatial split.
Generating 3839 patches (Optimized)...
Generating 6410 patches (Optimized)...
Train shapes: (3839, 25, 25, 25), Test shapes: (6410, 25, 25, 25)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training CARS_10_SG1_MSC with Data Augmentation for 100 epochs...
Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 16s 626ms/step - accuracy: 0.2642 - loss: 2.3355 - val_accuracy: 0.5813 - val_loss: 1.2564
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 13s 442ms/step - accuracy: 0.5827 - loss: 1.3280 - val_accuracy: 0.6656 - val_loss: 0.8919
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 542ms/step - accuracy: 0.6614 - loss: 1.0045 - val_accuracy: 0.8039 - val_loss: 0.5633
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 538ms/step - accuracy: 0.7704 - loss: 0.6528 - val_accuracy: 0.8930 - val_l

In [22]:
# %% [Cell 20] Experiment 11/16: GA-iPLS (SG1_MSC)
row = all_experiments['10_SG1_MSC'].iloc[2]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: GA-iPLS (10_SG1_MSC)
Preprocessing Pipeline: 10_SG1_MSC
Applying SG (w=30, p=2, d=1)...
Applying MSC...
Selecting 87 bands...
Using pixelsize=13 for 87 bands
Detected 16 classes for spatial split.
Generating 3913 patches (Optimized)...
Generating 6336 patches (Optimized)...
Train shapes: (3913, 13, 13, 87), Test shapes: (6336, 13, 13, 87)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training GA-iPLS_10_SG1_MSC with Data Augmentation for 100 epochs...
Epoch 1/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 16s 254ms/step - accuracy: 0.3572 - loss: 1.9463 - val_accuracy: 0.7525 - val_loss: 0.7242
Epoch 2/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 12s 206ms/step - accuracy: 0.7926 - loss: 0.6545 - val_accuracy: 0.8889 - val_loss: 0.3624
Epoch 3/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 213ms/step - accuracy: 0.8992 - loss: 0.3549 - val_accuracy: 0.9433 - val_loss: 0.1686
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 8s 213ms/step - accuracy: 0.9313 - loss: 0.2118 - val_accuracy: 

In [23]:
# %% [Cell 21] Experiment 12/16: GA-iPLS_BOSS (SG1_MSC)
row = all_experiments['10_SG1_MSC'].iloc[3]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)

# ============================================================
# Pipeline 4: 10_SG1_SVN (4 methods)
# ============================================================




Starting Experiment: GA-iPLS_BOSS (10_SG1_MSC)
Preprocessing Pipeline: 10_SG1_MSC
Applying SG (w=30, p=2, d=1)...
Applying MSC...
Selecting 55 bands...
Using pixelsize=19 for 55 bands
Detected 16 classes for spatial split.
Generating 3720 patches (Optimized)...
Generating 6529 patches (Optimized)...
Train shapes: (3720, 19, 19, 55), Test shapes: (6529, 19, 19, 55)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training GA-iPLS_BOSS_10_SG1_MSC with Data Augmentation for 100 epochs...
Epoch 1/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 18s 480ms/step - accuracy: 0.2211 - loss: 2.4333 - val_accuracy: 0.6339 - val_loss: 1.3578
Epoch 2/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 11s 362ms/step - accuracy: 0.6340 - loss: 1.1878 - val_accuracy: 0.8605 - val_loss: 0.5062
Epoch 3/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 7s 383ms/step - accuracy: 0.7999 - loss: 0.6361 - val_accuracy: 0.9282 - val_loss: 0.2530
Epoch 4/100
19/19 ━━━━━━━━━━━━━━━━━━━━ 7s 361ms/step - accuracy: 0.8904 - loss: 0.3415 - val_

In [24]:
# %% [Cell 22] Experiment 13/16: BOSS (SG1_SVN)
row = all_experiments['10_SG1_SVN'].iloc[0]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: BOSS (10_SG1_SVN)
Preprocessing Pipeline: 10_SG1_SVN
Applying SG (w=30, p=2, d=1)...
Applying SNV...
Selecting 9 bands...
Using pixelsize=25 for 9 bands
Detected 16 classes for spatial split.
Generating 2863 patches (Optimized)...
Generating 7386 patches (Optimized)...
Train shapes: (2863, 25, 25, 9), Test shapes: (7386, 25, 25, 9)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training BOSS_10_SG1_SVN with Data Augmentation for 100 epochs...
Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 452ms/step - accuracy: 0.2221 - loss: 2.3894 - val_accuracy: 0.5288 - val_loss: 1.3964
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 223ms/step - accuracy: 0.5399 - loss: 1.4546 - val_accuracy: 0.6911 - val_loss: 0.9670
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 245ms/step - accuracy: 0.6256 - loss: 1.1136 - val_accuracy: 0.7393 - val_loss: 0.7449
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 220ms/step - accuracy: 0.7122 - loss: 0.8668 - val_accuracy: 0.8366 - val_loss: 0

In [25]:
# %% [Cell 23] Experiment 14/16: CARS (SG1_SVN)
row = all_experiments['10_SG1_SVN'].iloc[1]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: CARS (10_SG1_SVN)
Preprocessing Pipeline: 10_SG1_SVN
Applying SG (w=30, p=2, d=1)...
Applying SNV...
Selecting 4 bands...
Using pixelsize=25 for 4 bands
Detected 16 classes for spatial split.
Generating 3912 patches (Optimized)...
Generating 6337 patches (Optimized)...
Train shapes: (3912, 25, 25, 4), Test shapes: (6337, 25, 25, 4)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training CARS_10_SG1_SVN with Data Augmentation for 100 epochs...
Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 279ms/step - accuracy: 0.2422 - loss: 2.4035 - val_accuracy: 0.4977 - val_loss: 1.6487
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 149ms/step - accuracy: 0.4323 - loss: 1.7951 - val_accuracy: 0.5552 - val_loss: 1.4001
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - accuracy: 0.5191 - loss: 1.5725 - val_accuracy: 0.5951 - val_loss: 1.1760
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - accuracy: 0.5880 - loss: 1.3078 - val_accuracy: 0.5422 - val_

In [26]:
# %% [Cell 24] Experiment 15/16: GA-iPLS (SG1_SVN)
row = all_experiments['10_SG1_SVN'].iloc[2]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: GA-iPLS (10_SG1_SVN)
Preprocessing Pipeline: 10_SG1_SVN
Applying SG (w=30, p=2, d=1)...
Applying SNV...
Selecting 96 bands...
Using pixelsize=13 for 96 bands
Detected 16 classes for spatial split.
Generating 3179 patches (Optimized)...
Generating 7070 patches (Optimized)...
Train shapes: (3179, 13, 13, 96), Test shapes: (7070, 13, 13, 96)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training GA-iPLS_10_SG1_SVN with Data Augmentation for 100 epochs...
Epoch 1/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 292ms/step - accuracy: 0.3218 - loss: 2.1356 - val_accuracy: 0.7830 - val_loss: 0.9047
Epoch 2/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 225ms/step - accuracy: 0.6990 - loss: 0.9622 - val_accuracy: 0.8491 - val_loss: 0.3987
Epoch 3/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 232ms/step - accuracy: 0.8319 - loss: 0.4801 - val_accuracy: 0.9500 - val_loss: 0.1909
Epoch 4/100
33/33 ━━━━━━━━━━━━━━━━━━━━ 7s 227ms/step - accuracy: 0.8991 - loss: 0.3276 - val_accuracy: 

In [27]:
# %% [Cell 25] Experiment 16/16: GA-iPLS_BOSS (SG1_SVN)
row = all_experiments['10_SG1_SVN'].iloc[3]
sel_vals, method, pipeline = parse_wst_row(row)
sel_idx = get_indices_from_values(sel_vals, all_wl)
train_and_evaluate(method, pipeline, sel_idx, X_raw, y_raw)

# ============================================================
# FCovSel Experiments (5 pipelines)
# ============================================================




Starting Experiment: GA-iPLS_BOSS (10_SG1_SVN)
Preprocessing Pipeline: 10_SG1_SVN
Applying SG (w=30, p=2, d=1)...
Applying SNV...
Selecting 7 bands...
Using pixelsize=25 for 7 bands
Detected 16 classes for spatial split.
Generating 3082 patches (Optimized)...
Generating 7167 patches (Optimized)...
Train shapes: (3082, 25, 25, 7), Test shapes: (7167, 25, 25, 7)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training GA-iPLS_BOSS_10_SG1_SVN with Data Augmentation for 100 epochs...
Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 349ms/step - accuracy: 0.1517 - loss: 2.4956 - val_accuracy: 0.3716 - val_loss: 1.8652
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 179ms/step - accuracy: 0.3280 - loss: 1.9416 - val_accuracy: 0.4932 - val_loss: 1.3208
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 185ms/step - accuracy: 0.4230 - loss: 1.6308 - val_accuracy: 0.6381 - val_loss: 1.0386
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 178ms/step - accuracy: 0.5205 - loss: 1.3229 - val_accuracy: 0.75

In [28]:
# %% [Cell 26] FCovSel Experiment 1/5: RAW
if fcovsel_df is not None:
    row = fcovsel_df[fcovsel_df['Pipeline'] == 'RAW'].iloc[0]
    sel_vals, _, pipeline = parse_wst_row(row)
    sel_idx = get_indices_from_values(sel_vals, all_wl)
    train_and_evaluate('FCovSel', pipeline, sel_idx, X_raw, y_raw)




Starting Experiment: FCovSel (RAW)
Selecting 19 bands...
Using pixelsize=25 for 19 bands
Detected 16 classes for spatial split.
Generating 4850 patches (Optimized)...
Generating 5399 patches (Optimized)...
Train shapes: (4850, 25, 25, 19), Test shapes: (5399, 25, 25, 19)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training FCovSel_RAW with Data Augmentation for 100 epochs...
Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 14s 513ms/step - accuracy: 0.2509 - loss: 2.2762 - val_accuracy: 0.4428 - val_loss: 1.5022
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 354ms/step - accuracy: 0.4589 - loss: 1.5754 - val_accuracy: 0.6920 - val_loss: 1.0573
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 390ms/step - accuracy: 0.6271 - loss: 1.1618 - val_accuracy: 0.7835 - val_loss: 0.6480
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 408ms/step - accuracy: 0.7301 - loss: 0.8360 - val_accuracy: 0.8788 - val_loss: 0.4002
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 403ms/step - accuracy: 0.8097 

In [29]:
# %% [Cell 27] FCovSel Experiment 2/5: SG MSC
if fcovsel_df is not None:
    row = fcovsel_df[fcovsel_df['Pipeline'] == 'SG MSC'].iloc[0]
    sel_vals, _, pipeline = parse_wst_row(row)
    sel_idx = get_indices_from_values(sel_vals, all_wl)
    train_and_evaluate('FCovSel', 'SG_MSC', sel_idx, X_raw, y_raw)




Starting Experiment: FCovSel (SG_MSC)
Preprocessing Pipeline: SG_MSC
Applying SG (w=30, p=2, d=0)...
Applying MSC...
Selecting 19 bands...
Using pixelsize=25 for 19 bands
Detected 16 classes for spatial split.
Generating 3676 patches (Optimized)...
Generating 6573 patches (Optimized)...
Train shapes: (3676, 25, 25, 19), Test shapes: (6573, 25, 25, 19)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training FCovSel_SG_MSC with Data Augmentation for 100 epochs...
Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 9s 539ms/step - accuracy: 0.1799 - loss: 2.5009 - val_accuracy: 0.3874 - val_loss: 1.8265
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 8s 337ms/step - accuracy: 0.3713 - loss: 1.8484 - val_accuracy: 0.5685 - val_loss: 1.4633
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 398ms/step - accuracy: 0.5073 - loss: 1.5276 - val_accuracy: 0.6289 - val_loss: 1.0984
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 404ms/step - accuracy: 0.5981 - loss: 1.2068 - val_accuracy: 0.7382 - val_loss: 0.8

In [30]:
# %% [Cell 28] FCovSel Experiment 3/5: SG SNV
if fcovsel_df is not None:
    row = fcovsel_df[fcovsel_df['Pipeline'] == 'SG SNV'].iloc[0]
    sel_vals, _, pipeline = parse_wst_row(row)
    sel_idx = get_indices_from_values(sel_vals, all_wl)
    train_and_evaluate('FCovSel', 'SG_SNV', sel_idx, X_raw, y_raw)




Starting Experiment: FCovSel (SG_SNV)
Preprocessing Pipeline: SG_SNV
Applying SG (w=30, p=2, d=0)...
Applying SNV...
Selecting 20 bands...
Using pixelsize=25 for 20 bands
Detected 16 classes for spatial split.
Generating 2873 patches (Optimized)...
Generating 7376 patches (Optimized)...
Train shapes: (2873, 25, 25, 20), Test shapes: (7376, 25, 25, 20)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training FCovSel_SG_SNV with Data Augmentation for 100 epochs...
Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 12s 658ms/step - accuracy: 0.1922 - loss: 2.8572 - val_accuracy: 0.3862 - val_loss: 2.2423
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 341ms/step - accuracy: 0.3520 - loss: 2.1661 - val_accuracy: 0.5251 - val_loss: 1.5798
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 428ms/step - accuracy: 0.4696 - loss: 1.7396 - val_accuracy: 0.6190 - val_loss: 1.2504
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 422ms/step - accuracy: 0.5360 - loss: 1.4444 - val_accuracy: 0.7213 - val_loss: 0.

In [31]:
# %% [Cell 29] FCovSel Experiment 4/5: SG1 MSC
if fcovsel_df is not None:
    row = fcovsel_df[fcovsel_df['Pipeline'] == 'SG1 MSC'].iloc[0]
    sel_vals, _, pipeline = parse_wst_row(row)
    sel_idx = get_indices_from_values(sel_vals, all_wl)
    train_and_evaluate('FCovSel', 'SG1_MSC', sel_idx, X_raw, y_raw)




Starting Experiment: FCovSel (SG1_MSC)
Preprocessing Pipeline: SG1_MSC
Applying SG (w=30, p=2, d=1)...
Applying MSC...
Selecting 19 bands...
Using pixelsize=25 for 19 bands
Detected 16 classes for spatial split.
Generating 3594 patches (Optimized)...
Generating 6655 patches (Optimized)...
Train shapes: (3594, 25, 25, 19), Test shapes: (6655, 25, 25, 19)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training FCovSel_SG1_MSC with Data Augmentation for 100 epochs...
Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 9s 540ms/step - accuracy: 0.2639 - loss: 2.3117 - val_accuracy: 0.6536 - val_loss: 1.2444
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 7s 345ms/step - accuracy: 0.6114 - loss: 1.2811 - val_accuracy: 0.7972 - val_loss: 0.7115
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 391ms/step - accuracy: 0.7475 - loss: 0.8036 - val_accuracy: 0.8898 - val_loss: 0.3777
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 388ms/step - accuracy: 0.8404 - loss: 0.5293 - val_accuracy: 0.9098 - val_loss: 

In [32]:
# %% [Cell 30] FCovSel Experiment 5/5: SG1 SNV
if fcovsel_df is not None:
    row = fcovsel_df[fcovsel_df['Pipeline'] == 'SG1 SNV'].iloc[0]
    sel_vals, _, pipeline = parse_wst_row(row)
    sel_idx = get_indices_from_values(sel_vals, all_wl)
    train_and_evaluate('FCovSel', 'SG1_SNV', sel_idx, X_raw, y_raw)




Starting Experiment: FCovSel (SG1_SNV)
Preprocessing Pipeline: SG1_SNV
Applying SG (w=30, p=2, d=1)...
Applying SNV...
Selecting 20 bands...
Using pixelsize=25 for 20 bands
Detected 16 classes for spatial split.
Generating 3142 patches (Optimized)...
Generating 7107 patches (Optimized)...
Train shapes: (3142, 25, 25, 20), Test shapes: (7107, 25, 25, 20)
Normalizing data (StandardScaler) - Fit on Train...
Normalization Complete.
Training FCovSel_SG1_SNV with Data Augmentation for 100 epochs...
Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 9s 586ms/step - accuracy: 0.2496 - loss: 2.3474 - val_accuracy: 0.5792 - val_loss: 1.3668
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 364ms/step - accuracy: 0.5698 - loss: 1.2927 - val_accuracy: 0.7748 - val_loss: 0.9188
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 406ms/step - accuracy: 0.6855 - loss: 0.9806 - val_accuracy: 0.8492 - val_loss: 0.5324
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 404ms/step - accuracy: 0.7845 - loss: 0.6947 - val_accuracy: 0.8798 - val_loss: 

In [33]:
# %% [Cell 31] Summary
print("\n" + "="*60)
print("All 21 Indian Pines experiments completed!")
print("="*60)
print("Results saved to CNN_Results_IP/ directory.")
print("\nWST Methods: BOSS, CARS, GA-iPLS, GA-iPLS_BOSS")
print("WST Pipelines: SG_MSC, SG_SVN, SG1_MSC, SG1_SVN")
print("\nFCovSel Pipelines: RAW, SG_MSC, SG_SNV, SG1_MSC, SG1_SNV")



All 21 Indian Pines experiments completed!
Results saved to CNN_Results_IP/ directory.

WST Methods: BOSS, CARS, GA-iPLS, GA-iPLS_BOSS
WST Pipelines: SG_MSC, SG_SVN, SG1_MSC, SG1_SVN

FCovSel Pipelines: RAW, SG_MSC, SG_SNV, SG1_MSC, SG1_SNV
